# 1-레이어 어텐션: QK 회로와 OV 회로 - OV 회로: 정보 이동

- Tutorial ID: `tut-2-1`
- Tutorial: 1-레이어 어텐션: QK 회로와 OV 회로
- Section ID: `s2-1-2`
- Section: OV 회로: 정보 이동


In [ ]:
# ============================================================
# 코드 읽는 법 — OV 회로: 정보 이동
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) 입력 데이터가 어떤 중간 변수들을 거쳐 최종 출력으로 변환되는지 shape 중심으로 추적
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

print("=" * 55)
print("OV 회로 (Output-Value Circuit) 분석")
print("=" * 55)

vocab_size = 6
d_model = 4
d_head = 2
vocab = ['A', 'B', 'C', 'D', 'E', 'F']

np.random.seed(1)
W_E = np.random.randn(vocab_size, d_model) * 0.5
W_U = np.random.randn(d_model, vocab_size) * 0.3
W_V = np.random.randn(d_model, d_head) * 0.3
W_O = np.random.randn(d_head, d_model) * 0.3

In [ ]:
# OV 회로 행렬

In [ ]:
W_OV = W_V @ W_O  # (d_model, d_model)
print(f"W_OV = W_V @ W_O shape: {W_OV.shape}")

# 확장된 OV 회로
C_OV = W_E @ W_OV @ W_U  # (vocab, vocab)
print(f"C_OV = W_E @ W_OV @ W_U shape: {C_OV.shape}")

print("
확장된 OV 행렬 (행=소스 토큰, 열=영향받는 로짓):")
print("각 원소: '토큰 i에 주목할 때, 토큰 j의 로짓 변화'")
print("     " + "  ".join(f"{v:4s}" for v in vocab))
for i, vi in enumerate(vocab):
    row = " ".join(f"{C_OV[i,j]:4.1f}" for j in range(vocab_size))
    print(f"  {vi}: {row}")

In [ ]:
# 복사(Copying) 동작 감지

In [ ]:
print("
" + "=" * 55)
print("복사 동작 감지 (고유값 분석)")
print("=" * 55)

# 복사 행렬 시뮬레이션 (복사 동작이 강한 헤드)
W_V_copy = np.eye(d_model, d_head) * 0.8 + np.random.randn(d_model, d_head) * 0.1
W_O_copy = np.eye(d_head, d_model) * 0.8 + np.random.randn(d_head, d_model) * 0.1

W_OV_copy = W_V_copy @ W_O_copy
C_OV_copy = W_E @ W_OV_copy @ W_U

# 고유값 분석
eigenvalues = np.linalg.eigvals(C_OV)
eigenvalues_copy = np.linalg.eigvals(C_OV_copy)

# 양의 실수 고유값 비율 (복사 강도 지표)
pos_real_eig = np.real(eigenvalues[np.real(eigenvalues) > 0.01])
pos_real_eig_copy = np.real(eigenvalues_copy[np.real(eigenvalues_copy) > 0.01])

print(f"랜덤 OV 회로:")
print(f"  모든 고유값: {np.round(np.sort(np.real(eigenvalues))[::-1][:6], 3)}")
print(f"  양의 고유값 수: {len(pos_real_eig)}/{vocab_size}")

print(f"
복사형 OV 회로:")
print(f"  모든 고유값: {np.round(np.sort(np.real(eigenvalues_copy))[::-1][:6], 3)}")
print(f"  양의 고유값 수: {len(pos_real_eig_copy)}/{vocab_size}")
print(f"  고유값의 트레이스(대각합) = {np.trace(C_OV_copy):.3f}")

In [ ]:
# 스킵-트라이그램 (Skip-Trigram)

In [ ]:
print("
" + "=" * 55)
print("스킵-트라이그램 (1-레이어 트랜스포머의 핵심)")
print("=" * 55)
print("""
스킵-트라이그램 형식: "A ... B C"
  - A: 초기 컨텍스트 토큰 (QK 회로가 주목)
  - B: 중간 토큰 (현재 위치)  
  - C: 예측된 다음 토큰 (OV 회로가 결정)

예시:
  "keep ... in mind" → C_QK가 'keep'을 찾고,
                        C_OV가 'mind'의 로짓을 높임

1-레이어 모델 = 바이그램 테이블 + 스킵-트라이그램 테이블의 앙상블

수식:
logit(x_0, ..., x_T, next) = W_U·W_E[x_T]  (바이그램)
  + Σ_h Σ_s<T A^h_{T,s} · C_OV^h[x_s, next]  (스킵-트라이그램)
""")

# 스킵-트라이그램 예시 계산
def softmax(x, axis=-1):
        x = x - np.max(x, axis=axis, keepdims=True)
    return np.exp(x) / np.sum(np.exp(x), axis=axis, keepdims=True)

# "A B C D" 시퀀스에서 D 다음 예측
seq = [0, 1, 2, 3]  # A, B, C, D
X = W_E[seq]
Q = X @ np.random.randn(d_model, d_head) * 0.3
K = X @ np.random.randn(d_model, d_head) * 0.3
attn_scores = Q @ K.T / np.sqrt(d_head)
mask = np.triu(np.full((4, 4), -1e9), k=1)
A = softmax(attn_scores + mask)

# D(마지막 토큰)의 어텐션 가중치
last_attn = A[-1]  # D가 각 이전 토큰에 주목하는 정도
print(f"'D' 토큰의 어텐션 가중치:")
for i, (tok, w) in enumerate(zip(['A','B','C','D'], last_attn)):
    bar = '█' * int(w * 20)
    print(f"  →{tok}: {bar} {w:.3f}")

print("
어텐션은 특정 이전 토큰을 선택하고,")
print("OV 회로는 그 토큰의 정보를 다음 위치로 '이동'시킵니다.")